# 🔬 Lab Module 29 — Agent Economics & ROI Analysis

**FinTech Corp — LoanBot v4.0 Business Case & Value Tracking**

Lab này build end-to-end economics framework: từ token cost-per-decision → scaling model → 3-year NPV → value dashboard.

---

## Mục tiêu Lab
1. **Token Cost Calculator** — tính cost-per-decision theo từng tier
2. **ROI Calculator** — compute payback period, 3-year ROI, NPV
3. **Scaling Model** — break-even analysis, marginal cost curves
4. **Sensitivity Analysis** — test assumptions với Monte Carlo simulation
5. **Value Dashboard** — leading + lagging KPI tracker
6. **Full Business Case** — integrate thành executive presentation data

In [ ]:
# Section 0: Setup
import math, random, statistics
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from enum import Enum

try:
    import matplotlib.pyplot as plt
    import numpy as np
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

print('✅ Dependencies loaded')
print('💰 Module 29: Agent Economics & ROI Analysis')

## Section 1: Token Cost Calculator

Tính cost-per-decision cho LoanBot v4.0 theo 3 decision tiers.

In [ ]:
# Section 1: Token Cost Calculator

class DecisionTier(Enum):
    CLEAR      = 'CLEAR'       # 80% volume — Haiku single pass
    STANDARD   = 'STANDARD'    # 17% volume — 3-agent Sonnet debate
    HIGH_VALUE = 'HIGH_VALUE'  # 3% volume  — 5-agent panel

@dataclass
class TierConfig:
    tier: DecisionTier
    volume_pct: float
    n_agents: int
    model: str
    avg_input_tokens: int
    avg_output_tokens: int
    overhead_factor: float

# Anthropic pricing per MTok
MODEL_PRICING = {
    'haiku':  {'input': 0.80,  'output': 4.00},
    'sonnet': {'input': 3.00,  'output': 15.00},
}

TIER_CONFIGS = [
    TierConfig(DecisionTier.CLEAR,      0.80, 1, 'haiku',  1200, 300,  1.1),
    TierConfig(DecisionTier.STANDARD,   0.17, 3, 'sonnet', 3000, 800,  1.3),
    TierConfig(DecisionTier.HIGH_VALUE, 0.03, 5, 'sonnet', 3500, 1000, 1.5),
]

USD_TO_VND = 25_000
MONTHLY_CASES = 50_000

def compute_tier_cost(cfg: TierConfig) -> dict:
    pricing = MODEL_PRICING[cfg.model]
    total_in  = cfg.avg_input_tokens  * cfg.n_agents * cfg.overhead_factor
    total_out = cfg.avg_output_tokens * cfg.n_agents * cfg.overhead_factor
    cost_usd  = (total_in * pricing['input'] + total_out * pricing['output']) / 1_000_000
    cost_vnd  = int(cost_usd * USD_TO_VND)
    return {'tier': cfg.tier.value, 'volume_pct': cfg.volume_pct,
            'model': cfg.model, 'n_agents': cfg.n_agents,
            'cost_usd': round(cost_usd, 5), 'cost_vnd': cost_vnd}

print('💰 LOANBOT COST-PER-DECISION ANALYSIS')
print('=' * 70)
print(f'{"Tier":<12} {"Vol%":>5} {"Cases":>7} {"Agents":>7} {"Model":<8} {"Cost/dec":>10} {"Monthly":>15}')
print('-' * 70)

total_monthly = 0
weighted_avg = 0
for cfg in TIER_CONFIGS:
    c = compute_tier_cost(cfg)
    cases = int(MONTHLY_CASES * cfg.volume_pct)
    monthly_cost = c['cost_vnd'] * cases
    total_monthly += monthly_cost
    print(f"{c['tier']:<12} {c['volume_pct']:>5.0%} {cases:>7,} {c['n_agents']:>7} {c['model']:<8} {c['cost_vnd']:>8,} VNĐ {monthly_cost:>13,} VNĐ")

weighted_avg = total_monthly // MONTHLY_CASES
print('=' * 70)
print(f'TOTAL: {MONTHLY_CASES:,} cases, Monthly API Cost: {total_monthly:,} VNĐ')
print(f'Weighted Average Cost/Decision: {weighted_avg:,} VNĐ')

## Section 2: ROI Calculator — NPV & Payback Period

In [ ]:
# Section 2: ROI & NPV Calculator

@dataclass
class MonthlyCashFlow:
    month: int
    staff_savings: int
    error_savings: int
    speed_value: int
    compliance_value: int
    api_cost: int
    infra_cost: int
    governance_cost: int
    capex: int = 0

    @property
    def total_benefit(self):
        return self.staff_savings + self.error_savings + self.speed_value + self.compliance_value
    @property
    def total_cost(self):
        return self.api_cost + self.infra_cost + self.governance_cost + self.capex
    @property
    def net(self):
        return self.total_benefit - self.total_cost

def build_monthly_model(months: int = 36) -> List[MonthlyCashFlow]:
    flows = []
    for m in range(months + 1):
        growth = 1.0 + 0.005 * min(m, 24)
        ramp   = 0.0 if m == 0 else (0.5 + 0.1 * min(m, 5) if m <= 5 else 1.0)
        capex  = 2_300_000_000 if m == 0 else 0
        flows.append(MonthlyCashFlow(
            month=m,
            staff_savings=int(13_125_000_000 * ramp * growth),
            error_savings=int(342_000_000_000 * ramp * growth),
            speed_value=int(2_000_000_000 * ramp * growth),
            compliance_value=int(800_000_000 * ramp * growth),
            api_cost=int(340_000_000 * ramp),
            infra_cost=500_000_000,
            governance_cost=150_000_000,
            capex=capex
        ))
    return flows

def compute_npv(flows: List[MonthlyCashFlow], annual_rate: float = 0.12) -> int:
    monthly_rate = (1 + annual_rate) ** (1/12) - 1
    return int(sum(f.net / (1 + monthly_rate) ** f.month for f in flows))

flows = build_monthly_model(36)
npv = compute_npv(flows)
investment = sum(f.capex for f in flows)
total_3yr_net = sum(f.net for f in flows)
avg_monthly = statistics.mean(f.net for f in flows[7:])  # Post-ramp average
payback_days = investment / avg_monthly * 30 if avg_monthly > 0 else 999

print('📊 3-YEAR FINANCIAL MODEL SUMMARY')
print('=' * 50)
print(f'Total Investment (Month 0):  {investment/1e9:>10.2f} tỷ VNĐ')
print(f'3-Year Cumulative Net:       {total_3yr_net/1e9:>10.1f} tỷ VNĐ')
print(f'NPV (12% discount rate):     {npv/1e9:>10.1f} tỷ VNĐ')
print(f'Payback Period:              {payback_days:>10.1f} ngày')

# Monthly cashflow table (first 8 months)
print('\n📅 Monthly Cash Flow (first 8 months):')
print(f'{"Mo":>3} {"Benefit (tỷ)":>14} {"Cost (tỷ)":>11} {"Net (tỷ)":>11} {"Cumulative":>13}')
cumulative = 0
for f in flows[:8]:
    cumulative += f.net
    tag = ' ← CapEx' if f.capex > 0 else ''
    print(f'{f.month:>3} {f.total_benefit/1e9:>14.2f} {f.total_cost/1e9:>11.2f} {f.net/1e9:>11.2f} {cumulative/1e9:>11.2f}{tag}')

## Section 3: Scaling Economics & Break-Even Analysis

In [ ]:
# Section 3: Scaling Economics

def human_team_cost(monthly_cases: int) -> int:
    CASES_PER_PERSON = 250
    COST_PER_PERSON  = 175_000_000
    people = math.ceil(monthly_cases / CASES_PER_PERSON)
    return people * COST_PER_PERSON

def ai_loanbot_cost(monthly_cases: int, cpd: int = 115) -> int:
    return 500_000_000 + 150_000_000 + (monthly_cases * cpd)

volumes = list(range(0, 55_000, 250))
human_costs = [human_team_cost(v) for v in volumes]
ai_costs    = [ai_loanbot_cost(v) for v in volumes]
breakeven   = next((v for v, h, a in zip(volumes, human_costs, ai_costs) if a <= h), None)

print('📈 BREAK-EVEN ANALYSIS')
print('=' * 60)
print(f'{"Volume":>12} {"Human Team":>18} {"LoanBot":>18} {"Winner":<10}')
print('-' * 60)
for v in [1000, 3000, 6000, 10000, 20000, 50000]:
    h, a = human_team_cost(v), ai_loanbot_cost(v)
    winner = '🤖 LoanBot' if a < h else '👥 Human'
    print(f'{v:>12,} {h:>18,} {a:>18,} {winner}')
print(f'\n✅ Break-even: ~{breakeven:,} cases/month')
print(f'   At 50,000 cases: LoanBot saves {(human_team_cost(50000)-ai_loanbot_cost(50000))/1e9:.2f} tỷ VNĐ/month vs human')

# Marginal cost
delta = 5_000
print(f'\n📉 Marginal Cost (adding {delta:,} cases):')
print(f'  Human team: +{(human_team_cost(50000+delta)-human_team_cost(50000))/1e9:.2f} tỷ VNĐ/month')
print(f'  LoanBot:    +{(ai_loanbot_cost(50000+delta)-ai_loanbot_cost(50000))/1e6:.1f} triệu VNĐ/month')

## Section 4: Monte Carlo Sensitivity Analysis

In [ ]:
# Section 4: Sensitivity Analysis

def scenario_annual_net(
    staff_savings_b: float = 13.125,
    error_savings_b: float = 342.0,
    api_factor: float = 1.0,
    accuracy_pct: float = 1.0
) -> float:
    monthly_benefit = (
        staff_savings_b * 1e9 +
        error_savings_b * 1e9 * accuracy_pct +
        2.0e9 + 0.8e9
    )
    monthly_cost = 340_000_000 * api_factor + 500_000_000 + 150_000_000 + 4_375_000_000
    return (monthly_benefit - monthly_cost) * 12 / 1e9

base = scenario_annual_net()

print('📊 DETERMINISTIC SENSITIVITY')
print(f'{"Scenario":<40} {"Annual (tỷ)":>12} {"Delta":>10}')
print('-' * 65)
for name, val in [
    ('Optimistic: Error savings +20%', scenario_annual_net(error_savings_b=410)),
    ('Base Case',                      base),
    ('Conservative: API cost ×2',      scenario_annual_net(api_factor=2.0)),
    ('Pessimistic: 0% accuracy gain',  scenario_annual_net(accuracy_pct=0.0)),
    ('Worst: Staff savings −50%',      scenario_annual_net(staff_savings_b=6.5, accuracy_pct=0.0)),
]:
    icon = '✅' if val > 0 else '❌'
    print(f'{icon} {name:<38} {val:>12.1f} {val-base:>+10.1f}')

# Monte Carlo
random.seed(42)
mc = [scenario_annual_net(
    staff_savings_b=random.gauss(13.125, 2.0),
    error_savings_b=random.gauss(342, 50),
    api_factor=random.uniform(0.8, 2.5),
    accuracy_pct=random.betavariate(5, 2)
) for _ in range(1000)]
mc.sort()
print(f'\n🎲 MONTE CARLO (1,000 scenarios):')
print(f'  P5:  {mc[50]:>8.1f} tỷ VNĐ/năm')
print(f'  P50: {mc[500]:>8.1f} tỷ VNĐ/năm')
print(f'  P95: {mc[950]:>8.1f} tỷ VNĐ/năm')
print(f'  P(positive): {sum(1 for r in mc if r > 0)/10:.1f}%')

## Section 5: Value Dashboard — KPI Tracker

In [ ]:
# Section 5: Value Dashboard

@dataclass
class KPI:
    name: str
    current: float
    target: float
    unit: str
    kpi_type: str
    lower_better: bool

    @property
    def on_track(self):
        return self.current <= self.target if self.lower_better else self.current >= self.target
    @property
    def variance_pct(self):
        return (self.current - self.target) / abs(self.target) * 100
    @property
    def health(self):
        if self.on_track: return 'GREEN'
        return 'RED' if abs(self.variance_pct) > 15 else 'YELLOW'

KPIS = [
    KPI('Cost/Decision (VNĐ)',     118,   125,   'VNĐ',    'leading', True),
    KPI('Type I Error Rate',       0.082, 0.100, '%',      'leading', True),
    KPI('Confidence P50',          0.78,  0.75,  'score',  'leading', False),
    KPI('PSI Drift Score',         0.08,  0.10,  'score',  'leading', True),
    KPI('Fairness Disparity',      0.87,  0.80,  'ratio',  'leading', False),
    KPI('Explanation Coverage',    1.00,  1.00,  '%',      'leading', False),
    KPI('30-Day Default Rate',     0.042, 0.050, '%',      'lagging', True),
    KPI('Monthly Net Savings (tỷ)',12.4,  12.0,  'tỷ',    'lagging', False),
    KPI('Customer NPS',            72,    70,    'score',  'lagging', False),
    KPI('Rural Approval Rate',     0.45,  0.55,  'ratio',  'lagging', False),
]

def render_dashboard(kpis, title=''):
    icons = {'GREEN': '🟢', 'YELLOW': '🟡', 'RED': '🔴'}
    print(f'\n📊 LOANBOT VALUE DASHBOARD {title}')
    for ktype in ['leading', 'lagging']:
        label = 'LEADING (real-time)' if ktype == 'leading' else 'LAGGING (monthly)'
        print(f'\n  --- {label} ---')
        print(f'  {"KPI":<28} {"Current":>9} {"Target":>9} {"Δ%":>8} {"Health":<8}')
        for k in [x for x in kpis if x.kpi_type == ktype]:
            print(f'  {icons[k.health]} {k.name:<26} {k.current:>9.3f} {k.target:>9.3f} {k.variance_pct:>+7.1f}% {k.health}')
    on_track = sum(1 for k in kpis if k.on_track)
    alerts = [k for k in kpis if k.health == 'RED']
    overall = '🟢 GREEN' if not alerts else '🔴 RED'
    print(f'\n  Overall: {overall} | On track: {on_track}/{len(kpis)}')
    if alerts: print(f'  🔔 Alerts: {[k.name for k in alerts]}')

render_dashboard(KPIS, '— LoanBot v4.0 Current')

# Simulate value erosion 3 months later
ERODED = [
    KPI('Cost/Decision (VNĐ)',     122,   125,   'VNĐ',    'leading', True),
    KPI('Type I Error Rate',       0.114, 0.100, '%',      'leading', True),
    KPI('Confidence P50',          0.68,  0.75,  'score',  'leading', False),
    KPI('PSI Drift Score',         0.22,  0.10,  'score',  'leading', True),
    KPI('Fairness Disparity',      0.84,  0.80,  'ratio',  'leading', False),
    KPI('Explanation Coverage',    0.97,  1.00,  '%',      'leading', False),
    KPI('30-Day Default Rate',     0.058, 0.050, '%',      'lagging', True),
    KPI('Monthly Net Savings (tỷ)',11.2,  12.0,  'tỷ',    'lagging', False),
    KPI('Customer NPS',            71,    70,    'score',  'lagging', False),
    KPI('Rural Approval Rate',     0.43,  0.55,  'ratio',  'lagging', False),
]
render_dashboard(ERODED, '— 3 Months Later (Drift Detected)')
print('\n💡 Recommendation: PSI=0.22 → approaching 0.25 trigger. Initiate M27 self-improvement cycle.')

## Section 6: Full Business Case Report

**TODO:** Tích hợp tất cả sections thành một `BusinessCaseReport.generate()` method.

In [ ]:
# Section 6: Full Business Case — TODO then Solution

class BusinessCaseReport:
    def generate(self) -> str:
        """TODO: Generate executive business case using all prior calculations."""
        # Steps:
        # 1. Run build_monthly_model() and compute_npv()
        # 2. Run scenario_annual_net() for base/optimistic/pessimistic
        # 3. Compute break-even using human_team_cost() vs ai_loanbot_cost()
        # 4. Get dashboard status from KPIS
        # 5. Format as executive-readable report
        raise NotImplementedError('TODO')

print('📝 BusinessCaseReport skeleton. Implement generate() using above utilities.')

In [ ]:
# SOLUTION

def generate_executive_report() -> str:
    flows    = build_monthly_model(36)
    npv      = compute_npv(flows)
    invest   = 2_300_000_000
    base_ann = scenario_annual_net()
    opt_ann  = scenario_annual_net(error_savings_b=410)
    pess_ann = scenario_annual_net(accuracy_pct=0.0)
    be       = next((v for v, h, a in zip(range(0,60000,250),
                     [human_team_cost(v) for v in range(0,60000,250)],
                     [ai_loanbot_cost(v) for v in range(0,60000,250)]) if a <= h), 6250)
    avg_monthly = statistics.mean(f.net for f in flows[7:])
    payback_d   = invest / avg_monthly * 30
    on_track    = sum(1 for k in KPIS if k.on_track)

    return f"""{'='*65}
BUSINESS CASE: LoanBot v4.0 — FinTech Corp
{'='*65}

EXECUTIVE SUMMARY (5 Numbers for CFO)
{'─'*40}
1. Investment:       {invest/1e9:.1f} tỷ VNĐ one-time
2. Annual Savings:   {base_ann:.0f} tỷ VNĐ/năm (steady state)
3. Payback Period:   {payback_d:.0f} ngày
4. NPV (12% WACC):   {npv/1e9:.0f} tỷ VNĐ over 3 years
5. Break-even Vol:   {be:,} cases/month (FinTech: 50,000)

SENSITIVITY ANALYSIS
{'─'*40}
  Optimistic:  {opt_ann:.0f} tỷ VNĐ/năm  ✅
  Base Case:   {base_ann:.0f} tỷ VNĐ/năm  ✅
  Pessimistic: {pess_ann:.0f} tỷ VNĐ/năm  ✅ (still positive!)

VALUE DASHBOARD: {on_track}/{len(KPIS)} KPIs on track

RECOMMENDATION: ✅ GO — Recommend immediate full deployment.
{'='*65}"""

print(generate_executive_report())